# SHIPP Stage 7 Preparation — OpenRouteService Smoke Test

**Purpose:** prove that Databricks can retrieve the ORS secret and make one real
Matrix API request.

This is **not** the production route-enrichment pipeline.

Production integration must consume eligible Silver listing/request pairs and
preserve the raw response before normalization.


In [0]:
# Confirm the Databricks secret exists without printing it.
ORS_SCOPE = "shipp"
ORS_KEY_NAME = "ors-api-key"

ors_api_key = dbutils.secrets.get(
    scope=ORS_SCOPE,
    key=ORS_KEY_NAME,
)

assert ors_api_key
print("ORS secret retrieved successfully.")


In [0]:
# Read coordinates from trusted Silver once Stage 5 has passed.
from pyspark.sql import functions as F

listing = (
    spark.table("bootcamp_students.shipp_silver.silver_listings")
    .filter(F.col("listing_id") == "demo-listing-001")
    .select("longitude", "latitude")
    .first()
)

request = (
    spark.table("bootcamp_students.shipp_silver.silver_requests")
    .filter(F.col("request_id") == "demo-request-001")
    .select("longitude", "latitude")
    .first()
)

assert listing is not None
assert request is not None

locations = [
    [float(listing["longitude"]), float(listing["latitude"])],
    [float(request["longitude"]), float(request["latitude"])],
]

print("Silver coordinates loaded.")


In [0]:
import requests

url = "https://api.heigit.org/openrouteservice/v2/matrix/driving-car"

payload = {
    "locations": locations,
    "sources": [0],
    "destinations": [1],
    "metrics": ["distance", "duration"],
    "units": "m",
}

headers = {
    "Authorization": ors_api_key,
    "Content-Type": "application/json",
}

response = requests.post(
    url,
    json=payload,
    headers=headers,
    timeout=15,
)

print("HTTP status:", response.status_code)
response.raise_for_status()

raw_response = response.json()

assert "distances" in raw_response
assert "durations" in raw_response

# Never print the API key.
print("ORS response contains distances and durations.")
display(raw_response)


## Evidence to retain

- HTTP status
- redacted request shape (no API key)
- raw JSON response
- distance field
- duration field

The next implementation step after this smoke test is to persist the raw response
into a Bronze route-response table before normalizing it into `silver_routes`.
